# Lab 6.1 &mdash; Your First Tool with @tool

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 3 &middot; Module 6 &mdash; Frameworks for Building AI Agents**

### What you'll do
- Wrap a function as a tool with the @tool decorator
- See the name, description (from the docstring) and args the model reads
- Call the tool with .invoke() and get a real result

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** the graded steps use a tiny **LangChain-shaped shim** (same names &amp; shapes as real LangChain) so they run offline &amp; deterministically. Advanced labs end with an **optional** cell that runs the **real** library.

**Reference:** [Module 6 slides &mdash; Defining a tool](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 6 labs](./index.html)

In [ ]:
# Setup -- run me first
import os
WORK = "/tmp/biaa-lab-06-01"
os.makedirs(WORK, exist_ok=True)
print("Working dir:", WORK)

## Concept
In Module 5 you built tools by hand as dicts. A framework does it for you: LangChain's **`@tool`**
decorator turns a plain function into a **Tool** with a **name**, a **description** (taken from the
**docstring** &mdash; the text the model reads to decide when to use it), an **args** schema, and an
**`.invoke()`** method. Same idea as before &mdash; one decorator instead of a dict.

In [ ]:
import ast, operator
# safe arithmetic: walk a parsed AST of numbers + (+ - * / ** unary-minus) -- never bare eval()
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}
def safe_calc(expr):
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp):
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    return ev(ast.parse(expr, mode="eval").body)

# --- LangChain-SHAPED shim: a tool has .name, .description (from the docstring), .args, .invoke() ---
import inspect
class Tool:
    def __init__(self, fn, name=None, description=None):
        self.fn = fn
        self.name = name or fn.__name__
        self.description = (description or inspect.getdoc(fn) or "").strip()
        self.args = list(inspect.signature(fn).parameters)
    def invoke(self, value):
        return self.fn(**value) if isinstance(value, dict) else self.fn(value)
    def __repr__(self):
        return "Tool(name=%r)" % self.name
def tool(fn):
    """The @tool decorator -- wrap a plain function into a Tool (mirrors langchain_core.tools.tool)."""
    return Tool(fn)

@tool
def greet(name):
    """Say hello to someone by name."""
    return "Hello, " + name + "!"
print("name:", greet.name, "| description:", greet.description, "| args:", greet.args)
print("greet.invoke('Ada') ->", greet.invoke("Ada"))

## Your Turn
Write two real tools &mdash; a **calculator** and a **lookup** &mdash; each with a clear docstring
(the description the model reads), then call them with `.invoke()`.

In [ ]:
@tool
def calculator(expression):
    ___    # TODO: a one-line docstring describing this tool (the model reads it)
    return ___    # TODO: compute the expression with the safe calculator

@tool
def lookup(key):
    ___    # TODO: a one-line docstring: look up a known fact by its key
    facts = {"capital of france": "Paris", "population of metropolis": "120000"}
    return ___    # TODO: return the fact for key (lowercased/stripped), else "unknown"

try:
    print('calculator.name :', calculator.name)
    print('calculator.args :', calculator.args)
    print('calculator.invoke("120000/2") =', calculator.invoke('120000/2'))
    print('lookup.invoke("capital of france") =', lookup.invoke('capital of france'))
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("calculator is a Tool named 'calculator'", lambda: calculator.name == "calculator")
expect_true("its args schema is ['expression']", lambda: calculator.args == ["expression"])
expect_true("calculator.invoke computes 120000/2 == 60000.0", lambda: calculator.invoke("120000/2") == 60000.0)
expect_true("lookup finds a known fact", lambda: lookup.invoke("capital of france") == "Paris")
expect_true("lookup returns 'unknown' for a missing key", lambda: lookup.invoke("price of gold") == "unknown")

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

---
### Key takeaway
`@tool` turns a function into a Tool the agent can call. The docstring is the description the model reads -- next we see why that description is the tool's real API.

[&#8592; All Module 6 labs](./index.html) &nbsp;&middot;&nbsp; [Module 6 slides](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>